# yfinance 펀더멘탈 데이터 수집 가이드

> 우리 프로젝트에서 멀티AI 페이지에 활용할 수 있는 **모든 펀더멘탈 데이터**를 확인합니다.
> 
> - 대상: AAPL (Apple) 예시
> - 목적: PER, ROE, 매출, 성장률, 재무제표 등 시각화 가능한 데이터 전체 확인

In [ ]:
import yfinance as yf
import pandas as pd

# 종목 선택
symbol = "AAPL"
stock = yf.Ticker(symbol)
print(f"=== {symbol} 데이터 로드 완료 ===")

## 1. 기업 개요 (Company Info)

`stock.info` — 가장 핵심. PER, PBR, ROE, 시가총액 등 한 번에 가져올 수 있음.

In [ ]:
info = stock.info

# 가치 평가 지표
valuation = {
    "PER (Trailing)": info.get("trailingPE"),
    "PER (Forward)": info.get("forwardPE"),
    "PBR": info.get("priceToBook"),
    "PSR": info.get("priceToSalesTrailing12Months"),
    "EV/EBITDA": info.get("enterpriseToEbitda"),
    "EV/Revenue": info.get("enterpriseToRevenue"),
    "PEG Ratio": info.get("pegRatio"),
}

# 수익성 지표
profitability = {
    "ROE": info.get("returnOnEquity"),
    "ROA": info.get("returnOnAssets"),
    "영업이익률": info.get("operatingMargins"),
    "순이익률": info.get("profitMargins"),
    "매출총이익률": info.get("grossMargins"),
    "EBITDA 마진": info.get("ebitdaMargins"),
}

# 성장성 지표
growth = {
    "매출 성장률": info.get("revenueGrowth"),
    "이익 성장률": info.get("earningsGrowth"),
    "분기 매출 성장률": info.get("quarterlyRevenueGrowth"),
    "분기 이익 성장률": info.get("quarterlyEarningsGrowth"),
}

# 시장 데이터
market = {
    "시가총액": f"${info.get('marketCap', 0):,.0f}",
    "현재가": info.get("currentPrice"),
    "52주 최고": info.get("fiftyTwoWeekHigh"),
    "52주 최저": info.get("fiftyTwoWeekLow"),
    "50일 이동평균": info.get("fiftyDayAverage"),
    "200일 이동평균": info.get("twoHundredDayAverage"),
    "베타": info.get("beta"),
    "배당수익률": info.get("dividendYield"),
    "배당금": info.get("dividendRate"),
}

# 재무 건전성
financial_health = {
    "부채비율": info.get("debtToEquity"),
    "유동비율": info.get("currentRatio"),
    "속성비율": info.get("quickRatio"),
    "총부채": f"${info.get('totalDebt', 0):,.0f}",
    "총현금": f"${info.get('totalCash', 0):,.0f}",
    "잉여현금흐름": f"${info.get('freeCashflow', 0):,.0f}",
    "영업현금흐름": f"${info.get('operatingCashflow', 0):,.0f}",
}

# 기업 정보
company = {
    "회사명": info.get("longName"),
    "섹터": info.get("sector"),
    "산업": info.get("industry"),
    "직원 수": f"{info.get('fullTimeEmployees', 0):,}명",
    "국가": info.get("country"),
    "웹사이트": info.get("website"),
}

print("=" * 60)
print(f"📊 {symbol} 펀더멘탈 데이터 요약")
print("=" * 60)

for title, data in [
    ("🏢 기업 정보", company),
    ("💰 가치 평가", valuation),
    ("📈 수익성", profitability),
    ("🚀 성장성", growth),
    ("🏦 재무 건전성", financial_health),
    ("📉 시장 데이터", market),
]:
    print(f"\n{title}")
    print("-" * 40)
    for k, v in data.items():
        if isinstance(v, float) and v is not None:
            if abs(v) < 1:  # 비율(%)
                print(f"  {k:20s}: {v:.2%}")
            else:
                print(f"  {k:20s}: {v:.2f}")
        else:
            print(f"  {k:20s}: {v}")

## 2. 손익계산서 (Income Statement)

분기별 매출, 영업이익, 순이익 → **매출/순이익 성장률 그래프** 그릴 수 있음

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# 분기별 손익계산서
quarterly_income = stock.quarterly_income_stmt
print("=== 분기별 손익계산서 (최근 4분기) ===")
print(f"사용 가능한 항목: {list(quarterly_income.index)[:15]}...")

# 핵심 항목 추출
key_items = ['Total Revenue', 'Operating Income', 'Net Income', 'Gross Profit', 'EBITDA']
for item in key_items:
    if item in quarterly_income.index:
        vals = quarterly_income.loc[item]
        print(f"\n{item}:")
        for date, val in vals.items():
            if pd.notna(val):
                print(f"  {date.strftime('%Y-Q%q') if hasattr(date, 'strftime') else date}: ${val/1e9:.2f}B")

In [ ]:
# 📊 매출 & 순이익 추이 시각화
fig, ax1 = plt.subplots(figsize=(10, 5))

revenue = quarterly_income.loc['Total Revenue'].dropna().sort_index() / 1e9
net_income = quarterly_income.loc['Net Income'].dropna().sort_index() / 1e9
quarters = [d.strftime('%Y-%m') for d in revenue.index]

# 매출 (막대그래프)
bars = ax1.bar(quarters, revenue.values, color='#3b82f6', alpha=0.7, label='Revenue ($B)')
ax1.set_ylabel('Revenue ($B)', color='#3b82f6')

# 순이익 (선 그래프)
ax2 = ax1.twinx()
ax2.plot(quarters, net_income.values, color='#22c55e', marker='o', linewidth=2, label='Net Income ($B)')
ax2.set_ylabel('Net Income ($B)', color='#22c55e')

ax1.set_title(f'{symbol} - Revenue & Net Income (Quarterly)', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# 범례
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()
print("→ 이 그래프를 멀티AI Fundamental 패널에 표시할 예정")

## 3. 재무상태표 (Balance Sheet)

자산, 부채, 자본 구조 → **부채비율/유동비율 추이** 시각화 가능

In [ ]:
# 분기별 재무상태표
bs = stock.quarterly_balance_sheet
print("=== 재무상태표 항목 ===")
print(f"전체 항목 수: {len(bs.index)}")

key_bs = ['Total Assets', 'Total Liabilities Net Minority Interest', 
          'Stockholders Equity', 'Current Assets', 'Current Liabilities',
          'Total Debt', 'Cash And Cash Equivalents']
for item in key_bs:
    if item in bs.index:
        vals = bs.loc[item].dropna()
        latest = vals.iloc[0] / 1e9
        print(f"  {item}: ${latest:.2f}B")

# 부채비율 추이
if 'Total Liabilities Net Minority Interest' in bs.index and 'Stockholders Equity' in bs.index:
    liabilities = bs.loc['Total Liabilities Net Minority Interest'].dropna().sort_index()
    equity = bs.loc['Stockholders Equity'].dropna().sort_index()
    debt_ratio = (liabilities / equity * 100).dropna()
    
    print(f"\n부채비율 추이:")
    for d, v in debt_ratio.items():
        print(f"  {d.strftime('%Y-%m')}: {v:.1f}%")

## 4. 현금흐름표 (Cash Flow)

영업/투자/재무 현금흐름 → **잉여현금흐름(FCF) 추이** 시각화

In [ ]:
# 현금흐름표
cf = stock.quarterly_cashflow
key_cf = ['Operating Cash Flow', 'Free Cash Flow', 'Capital Expenditure', 
          'Investing Cash Flow', 'Financing Cash Flow']

fig, ax = plt.subplots(figsize=(10, 5))
for item in key_cf:
    if item in cf.index:
        vals = cf.loc[item].dropna().sort_index() / 1e9
        ax.plot([d.strftime('%Y-%m') for d in vals.index], vals.values, marker='o', label=item)

ax.set_title(f'{symbol} - Cash Flow (Quarterly)', fontsize=14, fontweight='bold')
ax.set_ylabel('$B')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. 배당금 & 주식분할 이력

In [ ]:
# 배당금 이력
dividends = stock.dividends
print(f"=== 배당금 이력 (최근 10회) ===")
print(dividends.tail(10).to_string())

# 주식분할 이력
splits = stock.splits
print(f"\n=== 주식분할 이력 ===")
if len(splits) > 0:
    print(splits.to_string())
else:
    print("분할 이력 없음")

# 배당금 추이 그래프
if len(dividends) > 0:
    fig, ax = plt.subplots(figsize=(10, 3))
    recent_div = dividends.tail(20)
    ax.bar(range(len(recent_div)), recent_div.values, color='#f59e0b')
    ax.set_xticks(range(len(recent_div)))
    ax.set_xticklabels([d.strftime('%Y-%m') for d in recent_div.index], rotation=45, fontsize=7)
    ax.set_title(f'{symbol} - Dividend History', fontweight='bold')
    ax.set_ylabel('$/share')
    plt.tight_layout()
    plt.show()

## 6. mplfinance 캔들차트 + 거래량

`mplfinance` 라이브러리로 TradingView 스타일 캔들차트 렌더링

In [ ]:
import mplfinance as mpf

# 최근 6개월 일봉 데이터
df = stock.history(period="6mo", interval="1d")
print(f"=== {symbol} 최근 6개월 데이터: {len(df)}행 ===")
print(df.tail())

# TradingView 스타일 캔들차트
mc = mpf.make_marketcolors(up='#22c55e', down='#ef4444', edge='inherit', 
                            wick='inherit', volume='in')
s = mpf.make_mpf_style(marketcolors=mc, base_mpf_style='nightclouds')

mpf.plot(df, type='candle', style=s, volume=True, 
         title=f'\n{symbol} - 6 Month Candlestick Chart',
         figsize=(14, 7), panel_ratios=(3, 1),
         mav=(5, 20, 60))  # 5일/20일/60일 이동평균선

print("→ 5일(빨강), 20일(파랑), 60일(초록) 이동평균선 포함")

In [ ]:
# mplfinance - RSI + MACD 포함 차트
import numpy as np

# RSI 계산
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
rsi = 100 - (100 / (1 + rs))

# MACD 계산
ema12 = df['Close'].ewm(span=12).mean()
ema26 = df['Close'].ewm(span=26).mean()
macd = ema12 - ema26
signal = macd.ewm(span=9).mean()
histogram = macd - signal

# 추가 패널로 RSI + MACD 표시
apds = [
    mpf.make_addplot(rsi, panel=2, color='purple', ylabel='RSI'),
    mpf.make_addplot(macd, panel=3, color='blue', ylabel='MACD'),
    mpf.make_addplot(signal, panel=3, color='orange'),
    mpf.make_addplot(histogram, panel=3, type='bar', color='gray', alpha=0.5),
]

mpf.plot(df, type='candle', style=s, volume=True,
         addplot=apds,
         title=f'\n{symbol} - Full Technical Analysis',
         figsize=(14, 10), panel_ratios=(4, 1, 1.5, 1.5),
         mav=(20,))

print("→ 캔들차트 + 거래량 + RSI(14) + MACD(12,26,9)")

## 7. 애널리스트 추천 & 목표가

In [ ]:
# 애널리스트 추천
print("=== 애널리스트 추천 ===")
rec = stock.recommendations
if rec is not None and len(rec) > 0:
    print(rec.tail(10))
else:
    print("추천 데이터 없음")

# 목표가
print(f"\n=== 목표가 ===")
print(f"  평균 목표가: ${info.get('targetMeanPrice', 'N/A')}")
print(f"  최고 목표가: ${info.get('targetHighPrice', 'N/A')}")
print(f"  최저 목표가: ${info.get('targetLowPrice', 'N/A')}")
print(f"  중앙 목표가: ${info.get('targetMedianPrice', 'N/A')}")
print(f"  애널리스트 수: {info.get('numberOfAnalystOpinions', 'N/A')}명")
print(f"  추천: {info.get('recommendationKey', 'N/A')}")

## 8. 요약: yfinance에서 가져올 수 있는 전체 데이터 목록

| 카테고리 | 메서드/속성 | 시각화 용도 |
|----------|-----------|------------|
| **기업 개요** | `stock.info` | PER/PBR/ROE 카드, 기업 프로필 |
| **손익계산서** | `stock.quarterly_income_stmt` | 📊 매출/순이익 성장률 바+라인 차트 |
| **재무상태표** | `stock.quarterly_balance_sheet` | 부채비율/유동비율 추이 |
| **현금흐름표** | `stock.quarterly_cashflow` | FCF/영업CF 추이 라인차트 |
| **주가 히스토리** | `stock.history()` | 📈 캔들차트 + RSI + MACD |
| **배당금** | `stock.dividends` | 배당 이력 바차트 |
| **주식분할** | `stock.splits` | split 이벤트 표시 |
| **애널리스트** | `stock.recommendations` | 추천/목표가 게이지 |
| **실적 발표** | `stock.earnings_dates` | 실적 캘린더 |
| **기관 보유** | `stock.institutional_holders` | 기관 보유 비중 파이차트 |
| **내부자 거래** | `stock.insider_transactions` | 내부자 매수/매도 타임라인 |

### 우리 프로젝트 적용 계획

```
[즉시] yfinance → fundamental_daily 테이블 (PER, PBR, 시가총액 등)
[즉시] yfinance → 멀티AI Fundamental 차트 (매출/순이익 성장률)
[1주 뒤] yfinance → ML 피처 파이프라인 (95 → 113 피처)
[한계 시] FMP API $29/월로 전환
```